In [451]:
# import and setup a model
import torch
from ultralytics import YOLO
import numpy as np

# yolo = YOLO("yolov5n.pt")
# net = yolo.model
# net.info()

import torch
yolo = torch.hub.load('ultralytics/yolov5', 'yolov5n', pretrained=True)
net = yolo.model.model;

Using cache found in /home/geeth/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2026-3-16 Python-3.11.14 torch-2.10.0+cu128 CPU

Fusing layers... 
YOLOv5n summary: 213 layers, 1867405 parameters, 0 gradients, 4.5 GFLOPs
Adding AutoShape... 


In [452]:
# # Here are all functions used in the runtime - they appered as they in the __init__
net;

In [453]:
# sniff a runtime
execution = []
no_dup_execution = []

# to remove duplicates
seen = set()

def add_hook(module, name):
    def hook(module, input, output):
        execution.append(name)
    return hook

for name, m in net.named_modules():
    if isinstance(m, (torch.nn.Conv2d, torch.nn.BatchNorm2d, torch.nn.SiLU,
                      torch.nn.MaxPool2d, torch.nn.Upsample)):
        m.register_forward_hook(add_hook(m, name))

# a forward pass
x = torch.randn(1, 3, 640, 640)
net(x)

for layer in execution:
    if layer not in seen:
        no_dup_execution.append(layer)
        seen.add(layer)



print(execution)
duplicates = [x for x in execution if execution.count(x) > 1]
print(any(execution.count(x) > 1 for x in execution))
print(duplicates)
print(no_dup_execution)
len(no_dup_execution)


## We cant really build map using this as python may reuse functions and some layers are not defined using function

['model.0.conv', 'model.0.act', 'model.1.conv', 'model.1.act', 'model.2.cv1.conv', 'model.2.cv1.act', 'model.2.m.0.cv1.conv', 'model.2.m.0.cv1.act', 'model.2.m.0.cv2.conv', 'model.2.m.0.cv2.act', 'model.2.cv2.conv', 'model.2.cv2.act', 'model.2.cv3.conv', 'model.2.cv3.act', 'model.3.conv', 'model.3.act', 'model.4.cv1.conv', 'model.4.cv1.act', 'model.4.m.0.cv1.conv', 'model.4.m.0.cv1.act', 'model.4.m.0.cv2.conv', 'model.4.m.0.cv2.act', 'model.4.m.1.cv1.conv', 'model.4.m.1.cv1.act', 'model.4.m.1.cv2.conv', 'model.4.m.1.cv2.act', 'model.4.cv2.conv', 'model.4.cv2.act', 'model.4.cv3.conv', 'model.4.cv3.act', 'model.5.conv', 'model.5.act', 'model.6.cv1.conv', 'model.6.cv1.act', 'model.6.m.0.cv1.conv', 'model.6.m.0.cv1.act', 'model.6.m.0.cv2.conv', 'model.6.m.0.cv2.act', 'model.6.m.1.cv1.conv', 'model.6.m.1.cv1.act', 'model.6.m.1.cv2.conv', 'model.6.m.1.cv2.act', 'model.6.m.2.cv1.conv', 'model.6.m.2.cv1.act', 'model.6.m.2.cv2.conv', 'model.6.m.2.cv2.act', 'model.6.cv2.conv', 'model.6.cv2.act',

120

In [454]:
# layer_nums_back_c3 = use net
layer_nums_c3 = []
layer_nums_sppf = []

for i, layer in enumerate(net.model):
    name = layer.__class__.__name__.lower()
    if 'c3' in name:
        layer_nums_c3.append(i)
    if 'sppf' in name:
        layer_nums_sppf.append(i)

layer_nums_back_c3 = layer_nums_c3[:4]

print("C3 layers:", layer_nums_c3)
print("C3 back layers:", layer_nums_back_c3)
print("SPPF layers:", layer_nums_sppf)

C3 layers: [2, 4, 6, 8, 13, 17, 20, 23]
C3 back layers: [2, 4, 6, 8]
SPPF layers: [9]


In [455]:
def flatten_atomic(module, prefix=""):
    atomic_layers = []
    for name, sub in module.named_children():
        full_name = f"{prefix}.{name}" if prefix else name
        if list(sub.children()):
            atomic_layers.extend(flatten_atomic(sub, prefix=full_name))
        else:
            atomic_layers.append({"full_name": full_name, "model": sub})
    return atomic_layers

atomic_layers_unordered_incomplete = flatten_atomic(net.model)
atomic_layers_unordered_incomplete;

In [456]:
# reorder C3 block as we heard in sniffing
# policy: cv1 -> cv2 -> cv3 -> m to cv1 -> m -> cv2 -> cv3
def reorder_c3_blocks(layers):
    reordered = []
    c3_buffer = []
    for layer in layers:
        name = layer['full_name']
        ln = int(name.split('.')[0])
        if any(x in name for x in ['.cv1.', '.cv2.', '.cv3.', '.m.']) and ln in layer_nums_c3:
            c3_buffer.append(layer)
        else:
            if c3_buffer:
                cv1 = [l for l in c3_buffer if '.cv1.' in l['full_name'] and '.m.' not in l['full_name']]
                cv2 = [l for l in c3_buffer if '.cv2.' in l['full_name'] and '.m.' not in l['full_name']]
                cv3 = [l for l in c3_buffer if '.cv3.' in l['full_name'] and '.m.' not in l['full_name']]
                m   = [l for l in c3_buffer if '.m.' in l['full_name']]
                reordered.extend(cv1 + m + cv2 + cv3)
                c3_buffer = []
            reordered.append(layer)
    if c3_buffer:
        cv1 = [l for l in c3_buffer if '.cv1.' in l['full_name'] and '.m.' not in l['full_name']]
        cv2 = [l for l in c3_buffer if '.cv2.' in l['full_name'] and '.m.' not in l['full_name']]
        cv3 = [l for l in c3_buffer if '.cv3.' in l['full_name'] and '.m.' not in l['full_name']]
        m   = [l for l in c3_buffer if '.m.' in l['full_name']]
        reordered.extend(cv1 + m + cv2 + cv3)
    return reordered

atomic_layer_ordered = reorder_c3_blocks(atomic_layers_unordered_incomplete)
atomic_layer_ordered


# reorder SPPF block as we heard in sniffing
# policy: cv1 -> cv2 -> m to cv1 -> m -> cv2
def reorder_sppf_blocks(layers):
    reordered = []
    sppf_buffer = []
    for layer in layers:
        name = layer['full_name']
        ln = int(name.split('.')[0])
        if any(x in name for x in ['.cv1', '.cv2', '.m']) and ln in layer_nums_sppf:
            sppf_buffer.append(layer)
        else:
            if sppf_buffer:
                cv1 = [l for l in sppf_buffer if '.cv1' in l['full_name'] and '.m' not in l['full_name']]
                cv2 = [l for l in sppf_buffer if '.cv2' in l['full_name'] and '.m' not in l['full_name']]
                m   = [l for l in sppf_buffer if '.m' in l['full_name']]
                reordered.extend(cv1 + m + cv2)
                sppf_buffer = []
            reordered.append(layer)
    if sppf_buffer:
        cv1 = [l for l in sppf_buffer if '.cv1' in l['full_name'] and '.m' not in l['full_name']]
        cv2 = [l for l in sppf_buffer if '.cv.' in l['full_name'] and '.m' not in l['full_name']]
        m   = [l for l in sppf_buffer if '.m.' in l['full_name']]
        reordered.extend(cv1 + m + cv2)
    return reordered

atomic_layer_ordered = reorder_sppf_blocks(atomic_layer_ordered)
atomic_layer_ordered;

In [457]:
# add missing concat, res_add
# policy:
# add a {'full_name': 'x', 'model': Concat()}, // literary an x letter 
# 1. just before cv3.conv in c3 blocks
# 2. in sppf add two more pools (copy it and put them after the first pool - fullname contains pool)
# 3. just before cv2.conv in sppf blocks
# add a {'full_name': 'x', 'model': "ResAdd()"}, // literary an x letter
# one per number of bottlenecks
# just before previously add Concat if in c3_back (in the last one) in the middle put them betwen bottlenecks


from copy import deepcopy
from torch import nn

atomic_layers_ordered_completed = deepcopy(atomic_layer_ordered)


i = 0
while i < len(atomic_layers_ordered_completed):
    layer = atomic_layers_ordered_completed[i]
    name = layer['full_name'].lower()
    ln = int(name.split('.')[0])
    is_c3_cv3 = ('.cv3.conv' in name and ln in layer_nums_c3)
    is_sppf_cv2 = ('.cv2.conv' in name and ln in layer_nums_sppf)
    place_res = '.m' in name and '.cv2.act' in name and ln in layer_nums_back_c3
    if place_res:
        atomic_layers_ordered_completed.insert(i+1, {'full_name': 'x.resadd', 'model': "resadd"})
        i += 1
    if '.m' in name and ln in layer_nums_sppf:
        atomic_layers_ordered_completed.insert(i+1, deepcopy(layer))
        atomic_layers_ordered_completed.insert(i+2, deepcopy(layer))
        i += 2
    if (is_c3_cv3 or is_sppf_cv2) and ('.m' not in name):
        atomic_layers_ordered_completed.insert(i, {'full_name': 'x.concat', 'model': "concat"})
        i += 1
    i += 1

atomic_layers_ordered_completed

[{'full_name': '0.conv',
  'model': Conv2d(3, 16, kernel_size=(6, 6), stride=(2, 2), padding=(2, 2))},
 {'full_name': '0.act', 'model': SiLU(inplace=True)},
 {'full_name': '1.conv',
  'model': Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))},
 {'full_name': '1.act', 'model': SiLU(inplace=True)},
 {'full_name': '2.cv1.conv',
  'model': Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1))},
 {'full_name': '2.cv1.act', 'model': SiLU(inplace=True)},
 {'full_name': '2.m.0.cv1.conv',
  'model': Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))},
 {'full_name': '2.m.0.cv1.act', 'model': SiLU(inplace=True)},
 {'full_name': '2.m.0.cv2.conv',
  'model': Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))},
 {'full_name': '2.m.0.cv2.act', 'model': SiLU(inplace=True)},
 {'full_name': 'x.resadd', 'model': 'resadd'},
 {'full_name': '2.cv2.conv',
  'model': Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1))},
 {'full_name': '2.cv2.act', 'model': SiLU(inplace=True)},
 {'ful

In [458]:
atomic_layers = atomic_layers_ordered_completed;
len(atomic_layers)

142

In [459]:
from typing import List, Dict

def generate_config(layers: List[Dict]) -> List[List[int]]:
    config=[]
    for layer in layers:
        inner=[]
        model=layer['model']
        op_type='unknown'
        if isinstance(model,str):
            op_type=model.lower()
        else:
            cls_name=model.__class__.__name__.lower()
            if 'conv' in cls_name:op_type='conv'
            elif 'batchnorm' in cls_name:op_type='bn'
            elif 'relu' in cls_name or 'silu' in cls_name:op_type='activation'
            elif 'maxpool' in cls_name:op_type='pool'
            elif 'upsample' in cls_name:op_type='upsample'
            elif 'concat' in cls_name:op_type='concat'
            elif 'resadd' in cls_name:op_type='resadd'
            elif 'c3' in cls_name:op_type='c3'
            elif 'sppf' in cls_name:op_type='sppf'
            elif 'detect' in cls_name:op_type='detect'
        inner.append(op_type)
        if op_type=='conv':
            in_ch=model.in_channels
            out_ch=model.out_channels
            k_h,k_w=model.kernel_size if isinstance(model.kernel_size,tuple) else (model.kernel_size,model.kernel_size)
            s_h,s_w=model.stride if isinstance(model.stride,tuple) else (model.stride,model.stride)
            p_h,p_w=model.padding if isinstance(model.padding,tuple) else (model.padding,model.padding)
            bias=model.bias is not None
            inner.extend([in_ch,out_ch,k_h,k_w,s_h,s_w,p_h,p_w,bias])
        elif op_type=='bn':
            num_feat=model.num_features
            eps=model.eps
            momentum=model.momentum
            affine=model.affine
            track_stats=model.track_running_stats
            inner.extend([num_feat,eps,momentum,affine,track_stats])
        elif op_type=='activation':
            inplace=getattr(model,'inplace',False)
            inner.append(inplace)
        elif op_type=='pool':
            k_h,k_w=model.kernel_size if isinstance(model.kernel_size,tuple) else (model.kernel_size,model.kernel_size)
            s_h,s_w=model.stride if isinstance(model.stride,tuple) else (model.stride,model.stride)
            p_h,p_w=model.padding if isinstance(model.padding,tuple) else (model.padding,model.padding)
            d_h,d_w=model.dilation if isinstance(model.dilation,tuple) else (model.dilation,model.dilation)
            ceil_mode=model.ceil_mode
            inner.extend([k_h,k_w,s_h,s_w,p_h,p_w,d_h,d_w,ceil_mode])
        elif op_type=='upsample':
            scale_h,scale_w=model.scale_factor if isinstance(model.scale_factor,tuple) else (model.scale_factor,model.scale_factor)
            mode=model.mode
            inner.extend([scale_h,scale_w,mode])
        config.append(inner)
    return config

config=generate_config(atomic_layers_ordered_completed)
len(config)

142

In [460]:
# conv:                         [op_type, in_ch, out_ch, k_h, k_w, s_h, s_w, p_h, p_w, bias]
# bn:                           [op_type, num_features, eps, momentum, affine, track_running_stats]
# activation:                   [op_type, inplace]
# pool:                         [op_type, k_h, k_w, s_h, s_w, p_h, p_w, d_h, d_w, ceil_mode]
# upsample:                     [op_type, scale_h, scale_w, mode]
# concat/resadd/c3/sppf/detect: [op_type] (no params)
config

[['conv', 3, 16, 6, 6, 2, 2, 2, 2, True],
 ['activation', True],
 ['conv', 16, 32, 3, 3, 2, 2, 1, 1, True],
 ['activation', True],
 ['conv', 32, 16, 1, 1, 1, 1, 0, 0, True],
 ['activation', True],
 ['conv', 16, 16, 1, 1, 1, 1, 0, 0, True],
 ['activation', True],
 ['conv', 16, 16, 3, 3, 1, 1, 1, 1, True],
 ['activation', True],
 ['resadd'],
 ['conv', 32, 16, 1, 1, 1, 1, 0, 0, True],
 ['activation', True],
 ['concat'],
 ['conv', 32, 32, 1, 1, 1, 1, 0, 0, True],
 ['activation', True],
 ['conv', 32, 64, 3, 3, 2, 2, 1, 1, True],
 ['activation', True],
 ['conv', 64, 32, 1, 1, 1, 1, 0, 0, True],
 ['activation', True],
 ['conv', 32, 32, 1, 1, 1, 1, 0, 0, True],
 ['activation', True],
 ['conv', 32, 32, 3, 3, 1, 1, 1, 1, True],
 ['activation', True],
 ['resadd'],
 ['conv', 32, 32, 1, 1, 1, 1, 0, 0, True],
 ['activation', True],
 ['conv', 32, 32, 3, 3, 1, 1, 1, 1, True],
 ['activation', True],
 ['resadd'],
 ['conv', 64, 32, 1, 1, 1, 1, 0, 0, True],
 ['activation', True],
 ['concat'],
 ['conv', 64

In [461]:
# make weights list
weights=[]

state_dict = net.state_dict()
weights_list_unordered_incomplete = [{ "name": k, "weight": v.cpu()} for k, v in state_dict.items()]

name_to_weight={w["name"]:w["weight"] for w in weights_list_unordered_incomplete}
for layer in atomic_layers_ordered_completed:
    fname=layer["full_name"]
    inner=[]
    if fname!="x":
        prefix="model."+fname
        for k,v in name_to_weight.items():
            # skip BatchNorm bookkeeping variable
            if "num_batches_tracked" in k:
                continue
            if k.startswith(prefix):
                inner.append(v)
    weights.append(inner)

In [462]:
weights;

for i,layer in enumerate(weights):
    print(f"layer {i}:")
    if not layer:
        print("  []")
    else:
        for j,t in enumerate(layer):
            print(f"  tensor {j} shape={tuple(t.shape)}")    

layer 0:
  tensor 0 shape=(16, 3, 6, 6)
  tensor 1 shape=(16,)
layer 1:
  []
layer 2:
  tensor 0 shape=(32, 16, 3, 3)
  tensor 1 shape=(32,)
layer 3:
  []
layer 4:
  tensor 0 shape=(16, 32, 1, 1)
  tensor 1 shape=(16,)
layer 5:
  []
layer 6:
  tensor 0 shape=(16, 16, 1, 1)
  tensor 1 shape=(16,)
layer 7:
  []
layer 8:
  tensor 0 shape=(16, 16, 3, 3)
  tensor 1 shape=(16,)
layer 9:
  []
layer 10:
  []
layer 11:
  tensor 0 shape=(16, 32, 1, 1)
  tensor 1 shape=(16,)
layer 12:
  []
layer 13:
  []
layer 14:
  tensor 0 shape=(32, 32, 1, 1)
  tensor 1 shape=(32,)
layer 15:
  []
layer 16:
  tensor 0 shape=(64, 32, 3, 3)
  tensor 1 shape=(64,)
layer 17:
  []
layer 18:
  tensor 0 shape=(32, 64, 1, 1)
  tensor 1 shape=(32,)
layer 19:
  []
layer 20:
  tensor 0 shape=(32, 32, 1, 1)
  tensor 1 shape=(32,)
layer 21:
  []
layer 22:
  tensor 0 shape=(32, 32, 3, 3)
  tensor 1 shape=(32,)
layer 23:
  []
layer 24:
  []
layer 25:
  tensor 0 shape=(32, 32, 1, 1)
  tensor 1 shape=(32,)
layer 26:
  []
layer 

In [463]:
from graphviz import Digraph
def visualize_graph(graph, raw):
    g = Digraph("AtomicDAG", format="pdf")
    g.attr(rankdir="TB", nodesep="0.6", ranksep="0.9")
    g.attr(
        "node",
        shape="box",
        fontsize="14",
        fontname="Helvetica",
        style="rounded"
    )
    for i, layer in enumerate(raw):
        name = layer["full_name"]
        label = f"{i}\n{name}"
        g.node(str(i), label)
    for tar, srcs in enumerate(graph):
        for src in srcs:
            if src >= 0:
                g.edge(str(src), str(tar))

    return g

In [464]:
# creating the DAG

# part I: by default connect to the previous layer
graph=[[idx-1] for idx, _ in enumerate(atomic_layers_ordered_completed)]

In [465]:
def print_list(*lists):
    for i,items in enumerate(zip(*lists)):print("idx", i,":\n",*items)
# print_list(graph)

graphviz_graph = visualize_graph(graph, atomic_layers_ordered_completed)
graphviz_graph;

In [466]:
# part II: handle special cases in c3 and sppf blocks and head
num_bottlenecks = {2: 1, 4: 2, 6: 3, 8: 1}
def node_name(idx):
    if 0 <= idx < len(atomic_layers_ordered_completed):
        return atomic_layers_ordered_completed[idx]["full_name"]
    return f"<out:{idx}>"

# c3 blocks
for i,layer in enumerate(atomic_layers_ordered_completed):
    name=layer["full_name"]
    try:
        layer_num = int(name.split('.')[0]) 
    except ValueError:
        layer_num = layer_num
    if '.resadd' in name:
        c3_last_resadd = i
        graph[i].append(i-5)
    if (name.split('.')[1:]==["cv1", "conv"]):
        c3_start = i
    if '.m' in name and '.cv2' in name and '.act' in name:
        c3_last_boottleneck = i
    if (name.split('.')[1:]==["cv2", "conv"]):
        graph[i] = [c3_start-1]
    if '.concat' in name and layer_num in layer_nums_back_c3:
        graph[i].append(c3_last_resadd)
    if '.concat' in name and layer_num in layer_nums_c3 and not layer_num in layer_nums_back_c3:
        graph[i].append(c3_last_boottleneck)
    
# sppf blocks
for i,layer in enumerate(atomic_layers_ordered_completed):
    name=layer["full_name"]
    try:
        layer_num = int(name.split('.')[0]) 
    except ValueError:
        layer_num = layer_num 
    if '.concat' in name and layer_num in layer_nums_sppf:
        graph[i] = [i-1, i-2, i-3, i-4]
        graph[i+1] = [i]

# # disconnect the sequential connections in head
# for i,layer in enumerate(atomic_layers_ordered_completed):
#     name=layer["full_name"]
#     try:
#         layer_num = int(name.split('.')[0]) 
#     except ValueError:
#         layer_num = 10e12
#     if layer_num == 24:
#         graph[i] = []

In [467]:
visualize_graph(graph, atomic_layers_ordered_completed);

In [468]:
# part III: see what the pytorch says about edges
layer_graph = {}
for i, m in enumerate(net.model):
    if hasattr(m, 'f'):
        if isinstance(m.f, int):
            from_layers = [m.f]
        else:
            from_layers = m.f if len(m.f) > 0 else [-1]
    else:
        from_layers = [-1]
    layer_graph[i] = from_layers


# this helper do this when to it we pass the src, dest in torch layer number it connects
def connect(src, tar, graph, raw, disconnect=False): # disconnect then connect
    # interate through the raw = atomic_layers_ordered_completed once while doing so find 
    # 1. c1 = the last idx with the src
    # 2. c2 = the very first with the tar
    # then in the graph add the idx 1 to the graph[c2].append(c1)
    last_src = None
    first_tar = None
    for i, layer in enumerate(raw):
        name = layer["full_name"]
        ln = int(name.split('.')[0]) if name.split('.')[0] != 'x' else None
        # find last src
        if ln == src:
            last_src = i
        # find first tar
        if first_tar is None and ln == tar:
            first_tar = i
    if last_src is None:
        raise ValueError(f"Source layer {src} not found")
    if first_tar is None:
        raise ValueError(f"Target layer {tar} not found")
    if disconnect:
        graph[first_tar]= [last_src]
    else:
        print(f"Connecting last_src to first_tar): {node_name(last_src)} to {node_name(first_tar)}")
        graph[first_tar].append(last_src)
    return graph

def apply_layer_graph(layer_graph, graph, raw, except_tars = None):
    for tar in layer_graph.keys():
        if tar not in except_tars:
            srcs = layer_graph[tar]
            for src in srcs:
                if src < 0:
                    continue
                graph = connect(src, tar, graph, raw)
    return graph

# part III: handle the concats in the neck
apply_layer_graph(layer_graph, graph, atomic_layers_ordered_completed, except_tars=[24]);

# manually connect the heads
##110->##137
##124->##138
##138->##139
graph[137] = [110]
graph[138] = [124]
graph[139] = [138]

Connecting last_src to first_tar): 6.cv3.act to 12
Connecting last_src to first_tar): 4.cv3.act to 16
Connecting last_src to first_tar): 14.act to 19
Connecting last_src to first_tar): 10.act to 22


In [469]:
# g = visualize_graph(graph, atomic_layers_ordered_completed).render("model/atomic_dag", view=False, cleanup=True)
visualize_graph(graph, atomic_layers_ordered_completed);

In [ ]:
# now the model is succusessfully converted
# 1. Graph
graph;
import random
while True:
    random_idx = random.randint(0, len(graph)-1)
    if 0 <= random_idx < len(graph) and len(graph[random_idx]) >= 1:
        break

print(f"Example graph entry at index {random_idx}: {graph[random_idx]}")
print(f"Total number of nodes in the graph: {len(graph)}")

# 2. Config
config;

print(f"Example config entry at index {random_idx}: {config[random_idx]}")
print(f"Total number of entries in the config: {len(config)}")


# 3. Weights
weights;

print(f"Example weights entry at index {random_idx}: {weights[random_idx]}")
print(f"Total number of entries in the weights: {len(weights)}")

# 4. Classes
classes = list(net.names.values())
print(f"Classes: {classes}")

# export these as npy to models/
np.save("model/graph.npy", np.array(graph, dtype=object))
np.save("model/config.npy", np.array(config, dtype=object))
np.save("model/weights.npy", np.array(weights, dtype=object))
np.save("model/classes.npy", np.array(classes, dtype=object))

# export graph pdf to model/
g = visualize_graph(graph, atomic_layers_ordered_completed).render("model/atomic_dag", view=False, cleanup=True)

Example graph entry at index 31: [30]
Total number of nodes in the graph: 142
Example config entry at index 31: ['activation', True]
Total number of entries in the config: 142
Example weights entry at index 31: []
Total number of entries in the weights: 142
['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'o

In [471]:
net.names

{0: 'person',
 1: 'bicycle',
 2: 'car',
 3: 'motorcycle',
 4: 'airplane',
 5: 'bus',
 6: 'train',
 7: 'truck',
 8: 'boat',
 9: 'traffic light',
 10: 'fire hydrant',
 11: 'stop sign',
 12: 'parking meter',
 13: 'bench',
 14: 'bird',
 15: 'cat',
 16: 'dog',
 17: 'horse',
 18: 'sheep',
 19: 'cow',
 20: 'elephant',
 21: 'bear',
 22: 'zebra',
 23: 'giraffe',
 24: 'backpack',
 25: 'umbrella',
 26: 'handbag',
 27: 'tie',
 28: 'suitcase',
 29: 'frisbee',
 30: 'skis',
 31: 'snowboard',
 32: 'sports ball',
 33: 'kite',
 34: 'baseball bat',
 35: 'baseball glove',
 36: 'skateboard',
 37: 'surfboard',
 38: 'tennis racket',
 39: 'bottle',
 40: 'wine glass',
 41: 'cup',
 42: 'fork',
 43: 'knife',
 44: 'spoon',
 45: 'bowl',
 46: 'banana',
 47: 'apple',
 48: 'sandwich',
 49: 'orange',
 50: 'broccoli',
 51: 'carrot',
 52: 'hot dog',
 53: 'pizza',
 54: 'donut',
 55: 'cake',
 56: 'chair',
 57: 'couch',
 58: 'potted plant',
 59: 'bed',
 60: 'dining table',
 61: 'toilet',
 62: 'tv',
 63: 'laptop',
 64: 'mou